# Company A Churn Prediction PoC

Company A is an anonymous telecommunications provider with historical customer and usage data.

The objective of this notebook is to identify churn drivers, build a machine learning model to predict customer churn, and translate the findings into a business proposal focused on targeted retention actions.

2. Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

3. Load Data

4. Data Understanding

In [2]:
client = pd.read_csv("telecom/Client.csv")
record = pd.read_csv("telecom/Record.csv")

print("Client shape:", client.shape)
print("Record shape:", record.shape)

client.head()

Client shape: (100000, 50)
Record shape: (100000, 51)


,uniqsubs,actvsubs,new_cell,crclscod,asl_flag,totcalls,totmou,totrev,adjrev,adjmou,...,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays,Customer_ID
0,2,1,U,A,N,1652,4228.00000,1504.62,1453.44,4085.00,...,0.0,N,U,U,U,U,U,Y,361.0,1000001
1,1,1,N,EA,N,14654,26400.00000,2851.68,2833.88,26367.00,...,0.0,Z,U,U,U,U,U,Y,240.0,1000002
2,1,1,Y,C,N,7903,24385.05333,2155.91,1934.47,24303.05,...,0.0,N,U,Y,U,U,U,Y,1504.0,1000003
3,1,1,Y,B,N,1502,3065.00000,2000.90,1941.81,3035.00,...,0.0,U,Y,U,U,U,U,Y,1812.0,1000004
4,1,1,Y,A,N,4485,14028.00000,2181.12,2166.48,13965.00,...,0.0,I,U,U,U,U,U,Y,434.0,1000005


In [3]:
record.head()

,rev_Mean,mou_Mean,totmrc_Mean,da_Mean,ovrmou_Mean,ovrrev_Mean,vceovr_Mean,datovr_Mean,roam_Mean,change_mou,...,mou_opkv_Mean,mou_opkd_Mean,drop_blk_Mean,attempt_Mean,complete_Mean,callfwdv_Mean,callwait_Mean,churn,months,Customer_ID
0,23.9975,219.25,22.500,0.2475,0.00,0.0,0.0,0.0,0.0,-157.25,...,55.220000,0.0,1.333333,52.333333,45.000000,0.0,0.333333,1,61,1000001
1,57.4925,482.75,37.425,0.2475,22.75,9.1,9.1,0.0,0.0,532.25,...,169.343333,0.0,9.333333,263.333333,193.333333,0.0,5.666667,0,56,1000002
2,16.9900,10.25,16.990,0.0000,0.00,0.0,0.0,0.0,0.0,-4.25,...,0.233333,0.0,0.333333,9.000000,6.000000,0.0,0.000000,1,58,1000003
3,38.0000,7.50,38.000,0.0000,0.00,0.0,0.0,0.0,0.0,-1.50,...,5.450000,0.0,0.000000,3.666667,3.666667,0.0,0.000000,0,60,1000004
4,55.2300,570.50,71.980,0.0000,0.00,0.0,0.0,0.0,0.0,38.50,...,218.086667,0.0,10.333333,222.333333,137.000000,0.0,0.000000,0,57,1000005


In [4]:
client.info()
record.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 50 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   uniqsubs          100000 non-null  int64  
 1   actvsubs          100000 non-null  int64  
 2   new_cell          100000 non-null  object 
 3   crclscod          100000 non-null  object 
 4   asl_flag          100000 non-null  object 
 5   totcalls          100000 non-null  int64  
 6   totmou            100000 non-null  float64
 7   totrev            100000 non-null  float64
 8   adjrev            100000 non-null  float64
 9   adjmou            100000 non-null  float64
 10  adjqty            100000 non-null  int64  
 11  avgrev            100000 non-null  float64
 12  avgmou            100000 non-null  float64
 13  avgqty            100000 non-null  float64
 14  avg3mou           100000 non-null  int64  
 15  avg3qty           100000 non-null  int64  
 16  avg3rev           100

In [5]:
client.info()
record.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 50 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   uniqsubs          100000 non-null  int64  
 1   actvsubs          100000 non-null  int64  
 2   new_cell          100000 non-null  object 
 3   crclscod          100000 non-null  object 
 4   asl_flag          100000 non-null  object 
 5   totcalls          100000 non-null  int64  
 6   totmou            100000 non-null  float64
 7   totrev            100000 non-null  float64
 8   adjrev            100000 non-null  float64
 9   adjmou            100000 non-null  float64
 10  adjqty            100000 non-null  int64  
 11  avgrev            100000 non-null  float64
 12  avgmou            100000 non-null  float64
 13  avgqty            100000 non-null  float64
 14  avg3mou           100000 non-null  int64  
 15  avg3qty           100000 non-null  int64  
 16  avg3rev           100

In [6]:
print("Client duplicated Customer_ID:", client["Customer_ID"].duplicated().sum())
print("Record duplicated Customer_ID:", record["Customer_ID"].duplicated().sum())

Client duplicated Customer_ID: 0
Record duplicated Customer_ID: 0


5. Merge Datasets

In [7]:
df = record.merge(client, on="Customer_ID", how="left")

print("Merged shape:", df.shape)
df.head()

Merged shape: (100000, 100)


,rev_Mean,mou_Mean,totmrc_Mean,da_Mean,ovrmou_Mean,ovrrev_Mean,vceovr_Mean,datovr_Mean,roam_Mean,change_mou,...,dwllsize,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays
0,23.9975,219.25,22.500,0.2475,0.00,0.0,0.0,0.0,0.0,-157.25,...,A,0.0,N,U,U,U,U,U,Y,361.0
1,57.4925,482.75,37.425,0.2475,22.75,9.1,9.1,0.0,0.0,532.25,...,A,0.0,Z,U,U,U,U,U,Y,240.0
2,16.9900,10.25,16.990,0.0000,0.00,0.0,0.0,0.0,0.0,-4.25,...,A,0.0,N,U,Y,U,U,U,Y,1504.0
3,38.0000,7.50,38.000,0.0000,0.00,0.0,0.0,0.0,0.0,-1.50,...,D,0.0,U,Y,U,U,U,U,Y,1812.0
4,55.2300,570.50,71.980,0.0000,0.00,0.0,0.0,0.0,0.0,38.50,...,O,0.0,I,U,U,U,U,U,Y,434.0


In [8]:
missing_client_match = df["totrev"].isna().mean()
print("Percentage of records without client match:", missing_client_match)

Percentage of records without client match: 0.0


6. Target Variable Review

In [9]:
df["churn"].value_counts()

churn
0    50438
1    49562
Name: count, dtype: int64

In [23]:
df["churn"].value_counts(normalize=True)

churn
0    0.50438
1    0.49562
Name: proportion, dtype: float64

In [ ]:
df["churn"].value_counts(normalize=True).plot(kind="bar")
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Proportion")
plt.show()

In [12]:
id_columns = ["Customer_ID"]

8. EDA — Key Questions

EDA 1 — Churn?
A higher churn rate indicates revenue leakage and justifies a retention-focused proposal.

EDA 2 — Lower revenue customer churn?
Potenial Variables:

rev_Mean
totmrc_Mean
avg3rev
avgrev

In [13]:
churn_rate = df["churn"].mean()
print("Churn rate:", churn_rate)

Churn rate: 0.49562


EDA 3 — Fall related to churn?
Variables:

change_mou
change_rev

In [ ]:
df.groupby("churn")["rev_Mean"].mean().plot(kind="bar")
plt.title("Average Monthly Revenue by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Mean Monthly Revenue")
plt.show()

EDA 4 — Service trouble are associated with churn?
Variables:

drop_blk_Mean
drop_vce_Mean
blck_vce_Mean

In [ ]:
df.groupby("churn")["drop_blk_Mean"].mean().plot(kind="bar")
plt.title("Dropped or Blocked Calls by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Mean Dropped/Blocked Calls")
plt.show()

EDA 5 — Customer care indicates friction?
Variables:

custcare_Mean
cc_mou_Mean

In [ ]:
df.groupby("churn")["custcare_Mean"].mean().plot(kind="bar")
plt.title("Customer Care Calls by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Mean Customer Care Calls")
plt.show()

9. Business Problem Definition
EDA suggests that churn may be associated with changes in usage, revenue behavior, service quality indicators, and customer profile characteristics.

Business objective:
Reduce customer churn by identifying customers at high risk of leaving.

ML task:
Binary classification.

Target variable:
churn.

Business action:
Rank customers by predicted churn probability and prioritize retention campaigns for those with high churn risk and high customer value.

10. Feature Selection

In [20]:
target = "churn"

X = df.drop(columns=["churn", "Customer_ID"])
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

X shape: (100000, 98)
y shape: (100000,)
Numeric features: 77
Categorical features: 21


11. Preprocessing Pipeline

In [22]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

12. Train/Test Split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_test distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (80000, 98)
X_test shape: (20000, 98)
y_train distribution:
churn
0    0.504375
1    0.495625
Name: proportion, dtype: float64
y_test distribution:
churn
0    0.5044
1    0.4956
Name: proportion, dtype: float64


13. Baseline Model — Logistic Regression

In [25]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['rev_Mean', 'mou_Mean',
                                                   'totmrc_Mean', 'da_Mean',
                                                   'ovrmou_Mean', 'ovrrev_Mean',
                                                   'vceovr_Mean', 'datovr_Mean',
                                                   'roam_Mean', 'change_mou',
                                                   'change_rev',
                                                   'drop_vce_Mean',
                                                   'drop_dat_Mean',
                                                   'blck_vce_Mean'...
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['new_cell', 'crclscod',
                                                   'asl_flag',
                                                   'prizm_social_one', 'area',
                                                   'dualband', 'refurb_new',
                                                   'hnd_webcap', 'ownrent',
                                                   'dwlltype', 'marital',
                                                   'infobase', 'HHstatin',
                                                   'dwllsize', 'ethnic',
                                                   'kid0_2', 'kid3_5',
                                                   'kid6_10', 'kid11_15',
                                                   'kid16_17', 'creditcd'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [26]:
y_pred_lr = log_reg_model.predict(X_test)
y_proba_lr = log_reg_model.predict_proba(X_test)[:, 1]

print("Model: Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1-score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

Model: Logistic Regression
Accuracy: 0.5956
Precision: 0.5886986967516048
Recall: 0.6106739305891848
F1-score: 0.5994849955432306
ROC-AUC: 0.6310252865981941
